# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/space-0d/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane (Refresh / Content Opportunity Scoring) is fundamentally a classification task — predicting whether a page is declining — whose output (a predicted probability) is used as a ranking score to build the review queue. It's not clustering (I'm not grouping similar pages) and not pure unsupervised scoring (there's a real binary label underneath). The classifier's probability output is what gets ranked, and reason codes come from the model's feature importances / decision paths.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is is_declining_label, defined as trend_direction == "down" — i.e. impressions in the last 30 days dropped more than 20% versus the prior 30 days. This is a rule-defined label, but it's a rule applied to observed behavioral data (real GSC impression trends), not a subjective hand-tuned score. Important: trend_direction and trend_pct are the label source, so they're excluded from my model features to avoid leakage.



In [1]:

import pandas as pd
import glob


matches = glob.glob("/content/**/content_refresh_anonymized.csv", recursive=True)

if matches:
    df = pd.read_csv(matches[0])
    print(f"Loaded from: {matches[0]}")
else:
    print("File not found — upload it directly:")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df.shape)
df["is_declining_label"].value_counts(normalize=True)


File not found — upload it directly:


Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
(30000, 45)


,proportion
is_declining_label,
1,0.542067
0,0.457933


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K (Precision@50) — of the top 50 pages my model ranks as declining, what fraction actually are. This matches how the output is used: someone works down a ranked queue, not the whole dataset. The reference pipeline's baseline hand-rule scores ~0.24 Precision@50; a trained model should clear that meaningfully.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (content_id), pseudonymized, with 90-day aggregated GSC/GA4 metrics plus trend and tier columns.

In [2]:

lane_cols = [
    "content_id", "client_id",
    "search_volume", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "position_tier", "impression_tier",
    "trend_direction", "trend_pct",
    "content_type", "main_intent",
    "days_since_last_update", "freshness_tier",
]
df[lane_cols].head()


,content_id,client_id,search_volume,impressions_90d,clicks_90d,ctr,avg_position,position_tier,impression_tier,trend_direction,trend_pct,content_type,main_intent,days_since_last_update,freshness_tier
0,content_304f48230142,client_f369cb89fc,10.0,3803,29,0.76,10.6,striking,good,down,-41.4,keyword article,transactional,20,0-30
1,content_a1fb4e703a9e,client_4e07408562,90.0,15320,7,0.05,20.3,page_3_5,good,down,-57.7,keyword article,informational,25,0-30
2,content_9aa793d4d895,client_7f2253d7e2,0.0,12581,11,0.09,36.5,page_3_5,good,down,-60.9,keyword article,informational,20,0-30
3,content_331d6c4de07b,client_19581e27de,10.0,11751,58,0.49,6.2,page_1,good,stable,-13.8,keyword article,commercial,22,0-30
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,19140,24,0.13,44.0,page_3_5,good,down,-34.7,keyword article,informational,14,0-30


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A hand-rule has to fix its thresholds and weights up front — e.g. "low CTR + deep position = declining." But real pages decline for different, interacting reasons: low CTR at top_3 position points to a title/snippet problem, while the same CTR at deep position points to a relevance or content-quality problem. A trained classifier learns which combinations matter and gets validated on held-out clients via Precision@K, instead of trusting hand-picked thresholds that don't adapt as the data shifts.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.